In [1]:
# ===================== ALL-IN-ONE: 3D simplex (tetrahedra) for PID-SP =====================
# Requirements: pyomo, gurobi, numpy, scipy, plotly, tqdm, csv file "data.csv"
# -----------------------------------------------------------------------------------------

import numpy as np
import itertools as it
import csv
from tqdm import tqdm
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
from scipy.spatial import Delaunay
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# ------------------------- Config knobs -------------------------
MIN_DIST   = 1e-8     # 去重阈值
ACTIVE_TOL = 1e-8     # active 判定容差
MS_AGG     = "sum"    # 单形 ms 聚合：'sum' 或 'mean'

# ------------------------- PID scenario model -------------------------
def build_pid_model(T=10, h=0.2, scen=None, weights=(1.0, 0.01),
                    bounds=None, use_cvar=False, alpha=0.95):
    assert scen is not None, "请提供一个场景字典"
    Ku, tau, d, sp = scen["Ku"], scen["tau"], scen["d"], scen["sp"]
    assert len(d) == T+1 and len(sp) == T+1

    if bounds is None:
        bounds = {}
    bx = bounds.get("x",  (-20, 20))
    bu = bounds.get("u",  (None, None))
    bKp= bounds.get("Kp", (0, 10))
    bKi= bounds.get("Ki", (0, 10))
    bKd= bounds.get("Kd", (0, 10))
    be = bounds.get("e",  (-100, 100))
    bI = bounds.get("I",  (-200, 200))

    m = pyo.ConcreteModel()
    m.T  = pyo.RangeSet(0, T)
    m.Tm = pyo.RangeSet(1, T)

    m.Kp = pyo.Var(bounds=bKp)
    m.Ki = pyo.Var(bounds=bKi)
    m.Kd = pyo.Var(bounds=bKd)

    m.x = pyo.Var(m.T, bounds=bx)
    m.u = pyo.Var(m.T, bounds=bu)
    m.e = pyo.Var(m.T, bounds=be)
    m.I = pyo.Var(m.T, bounds=bI)

    def _err_rule(m, t): return m.e[t] == sp[t] - m.x[t]
    m.err_def = pyo.Constraint(m.T, rule=_err_rule)

    def _I_dyn(m, t): return m.I[t] == m.I[t-1] + h*m.e[t]
    m.I_dyn = pyo.Constraint(m.Tm, rule=_I_dyn)

    def _x_dyn(m, t):
        return m.x[t] == m.x[t-1] + (h/tau)*(-m.x[t] + Ku*m.u[t] + d[t])
    m.x_dyn = pyo.Constraint(m.Tm, rule=_x_dyn)

    def _pid_rule(m, t):
        if t == 0:
            return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t]
        return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t] + m.Kd*(m.e[t]-m.e[t-1])/h
    m.pid = pyo.Constraint(m.T, rule=_pid_rule)

    m.x0 = pyo.Constraint(expr=m.x[0] == 0)
    m.I0 = pyo.Constraint(expr=m.I[0] == 0)

    w_e, w_u = weights
    m.cost = pyo.Expression(expr=sum(h*(w_e*m.e[t]**2 + w_u*m.u[t]**2) for t in m.T))
    m.obj_expr = pyo.Expression(expr=m.cost)
    return m, [m.Kp, m.Ki, m.Kd]

def load_scenarios_from_csv(csv_path: str, T: int | None = None,
                            sp0: float = 0.0, sp1: float = 0.5,
                            ku_col: str = "tau_us", tau_col: str = "tau_xs",
                            disturb_prefix: str = "disturbance_",
                            setpoint_change_col: str = "setpoint_change"):
    scens = []
    # 推断 T
    if T is None:
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fields  = reader.fieldnames or []
            max_idx = -1
            for name in fields:
                if name.startswith(disturb_prefix):
                    try:
                        k = int(name[len(disturb_prefix):])
                        max_idx = max(max_idx, k)
                    except:
                        pass
            if max_idx < 0:
                raise ValueError(f"未找到扰动列前缀 {disturb_prefix}k")
            T = max_idx

    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            Ku  = float(row[ku_col])
            tau = float(row[tau_col])

            d = []
            for t in range(T+1):
                col = f"{disturb_prefix}{t}"
                d.append(float(row[col]))

            sp = [sp1]*(T+1)
            if setpoint_change_col in row and row[setpoint_change_col] != "":
                try:
                    t_star = int(float(row[setpoint_change_col]))
                    for t in range(T+1):
                        sp[t] = sp0 if t < t_star else sp1
                except:
                    pass

            scens.append({"Ku": Ku, "tau": tau, "d": d, "sp": sp})
    return scens, T

def build_models_from_csv(csv_path: str, h: float = 0.2,
                          weights=(1.0, 0.01), bounds=None,
                          sp0: float = 0.0, sp1: float = 0.5,
                          ku_col: str = "tau_us", tau_col: str = "tau_xs",
                          disturb_prefix: str = "disturbance_",
                          setpoint_change_col: str = "setpoint_change",
                          max_scenarios=None, skip=0):
    scens, T = load_scenarios_from_csv(
        csv_path=csv_path, T=None, sp0=sp0, sp1=sp1,
        ku_col=ku_col, tau_col=tau_col,
        disturb_prefix=disturb_prefix, setpoint_change_col=setpoint_change_col,
    )
    if skip or max_scenarios:
        scens = scens[skip: (skip + max_scenarios) if max_scenarios else None]

    model_list, first_stg_vars_list = [], []
    for scen in scens:
        m, yvars = build_pid_model(T=T, h=h, scen=scen, weights=weights, bounds=bounds)
        model_list.append(m)
        first_stg_vars_list.append(yvars)

    m_tmpl_list = [model_list[0], first_stg_vars_list[0]]
    return model_list, first_stg_vars_list, m_tmpl_list, T

# ------------------------- Basic utils -------------------------
def corners_from_var_bounds(vars_3):
    bnds = []
    for v in vars_3:
        lb, ub = v.lb, v.ub
        if lb is None or ub is None:
            raise ValueError(f"{v.name} 缺少上下界")
        bnds.append((float(lb), float(ub)))
    return [tuple(p) for p in it.product(*[(lo, hi) for (lo,hi) in bnds])]

def too_close(p, nodes, tol=MIN_DIST):
    return any(np.linalg.norm(np.asarray(p)-np.asarray(q)) < tol for q in nodes)

def evaluate_Q_at(model, first_stg_vars, first_stg_vals, solver):
    """把 Kp,Ki,Kd 固定为给定值，最小化 obj_expr，返回该场景下的真实目标值（用 Gurobi 解）。"""
    if hasattr(model, 'obj'):
        model.del_component('obj')
    for u, v in zip(first_stg_vars, first_stg_vals):
        u.fix(float(v))
    model.obj = pyo.Objective(expr=model.obj_expr, sense=pyo.minimize)
    try:
        results = solver.solve(model, tee=False)
        ok = (results.solver.status == SolverStatus.ok) and \
             (results.solver.termination_condition == TerminationCondition.optimal)
        if not ok:
            raise RuntimeError(f"evaluate_Q_at not optimal: {results.solver.status}, {results.solver.termination_condition}")
        return float(pyo.value(model.obj_expr))
    finally:
        if hasattr(model, 'obj'):
            model.del_component('obj')
        for u in first_stg_vars:
            if u.fixed:
                u.unfix()

# ------------------------- Point selection inside a simplex -------------------------
def _aggregate_scenario_values(vals_scn, mode="mean", weights=None, cvar_alpha=0.9):
    vals = np.asarray(vals_scn, float)
    if vals.ndim == 0:
        return float(vals)
    w = None
    if weights is not None:
        w = np.asarray(weights, float)
        if w.sum() > 0:
            w = w / w.sum()
        else:
            w = None
    if mode == "mean":
        return float(np.average(vals, weights=w))
    elif mode == "worst":
        return float(np.max(vals))
    elif mode == "best":
        return float(np.min(vals))
    elif mode == "median":
        return float(np.median(vals))
    elif mode == "cvar":
        sample = vals
        if w is not None:
            k = max(1000, 10*len(vals))
            idx = np.random.choice(len(vals), size=k, p=w)
            sample = vals[idx]
        q = np.quantile(sample, cvar_alpha)
        tail = sample[sample >= q]
        return float(tail.mean() if len(tail) else q)
    return float(np.average(vals, weights=w))

def _simplex_barycenter(pts4):
    return np.mean(pts4, axis=0)

def _simplex_incenter_approx(pts4):
    faces = [(1,2,3),(0,2,3),(0,1,3),(0,1,2)]
    fc = [np.mean(pts4[list(f)], axis=0) for f in faces]
    return np.mean(np.vstack(fc), axis=0)

def _project_to_simplex(x, pts4):
    T = np.vstack((pts4[1:] - pts4[0]))
    try:
        lam = np.linalg.lstsq(T.T, (x - pts4[0]), rcond=None)[0]
    except np.linalg.LinAlgError:
        return _simplex_barycenter(pts4)
    lams = np.concatenate(([1 - lam.sum()], lam))
    lams = np.clip(lams, 0.0, 1.0)
    s = lams.sum()
    if s <= 0:
        return _simplex_barycenter(pts4)
    lams /= s
    return (lams[:,None] * pts4).sum(axis=0)

def _farthest_point_approx(pts4, existing_pts, n_rand=12, rng=None):
    rng = np.random.default_rng(None if rng is None else rng)
    cands = []
    cands.append(_simplex_barycenter(pts4))
    cands.append(_simplex_incenter_approx(pts4))
    faces = [(1,2,3),(0,2,3),(0,1,3),(0,1,2)]
    for f in faces:
        cands.append(np.mean(pts4[list(f)], axis=0))
    R = rng.random((n_rand,4))
    R = R / R.sum(axis=1, keepdims=True)
    rand_pts = (R[:,:,None] * pts4[None,:,:]).sum(axis=1)
    cands.extend(list(rand_pts))
    X = np.asarray(existing_pts, float)
    best_x, best_d = None, -np.inf
    for x in cands:
        d = np.inf if len(X)==0 else np.linalg.norm(X - x, axis=1).min()
        if d > best_d:
            best_d, best_x = d, x
    return best_x

def choose_point_in_simplex(
    simplex_id,
    simplices,
    points,
    scen_values_per_point,          # (n_points, S)
    mode="incenter",                # "incenter" | "barycenter" | "farthest" | "face_centroid"
    min_dist=1e-8,
    risk_mode="mean",               # 多场景聚合：mean/worst/best/median/cvar
    scenario_weights=None,
    cvar_alpha=0.9,
    rng=None,
):
    """
    在给定单形内部选择下一个插值点（多场景兼容）。
    必要输入：
      - points: (n_points, 3)
      - scen_values_per_point: (n_points, S)
    返回：new_point (3,)
    """
    pts = np.asarray(points, float)
    Svals = np.asarray(scen_values_per_point, float)  # (N,S)
    simplices = np.asarray(simplices, int)

    verts_idx = simplices[simplex_id]
    pts4 = pts[verts_idx]            # (4,3)
    vvals = Svals[verts_idx, :]      # (4,S)

    if mode == "incenter":
        x = _simplex_incenter_approx(pts4)
    elif mode == "barycenter":
        x = _simplex_barycenter(pts4)
    elif mode == "face_centroid":
        agg = [ _aggregate_scenario_values(vvals[i], mode=risk_mode,
                                           weights=scenario_weights, cvar_alpha=cvar_alpha)
                for i in range(4) ]
        i_min = int(np.argmin(agg))
        face = [j for j in range(4) if j != i_min]
        x = np.mean(pts4[face], axis=0)
    elif mode == "farthest":
        x = _farthest_point_approx(pts4, pts, rng=rng)
    else:
        x = _simplex_incenter_approx(pts4)

    x = _project_to_simplex(x, pts4)

    def _too_close(xx): 
        return len(pts) and (np.linalg.norm(pts - xx, axis=1).min() < float(min_dist))

    if _too_close(x):
        backups = [
            _simplex_barycenter(pts4),
            _farthest_point_approx(pts4, pts, rng=rng),
        ]
        for b in backups:
            b = _project_to_simplex(b, pts4)
            if not _too_close(b):
                return tuple(map(float, b))
        edges = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
        lens = [np.linalg.norm(pts4[i]-pts4[j]) for (i,j) in edges]
        i = int(np.argmax(lens)); a,b = edges[i]
        x = 0.5*(pts4[a] + pts4[b])
        x = _project_to_simplex(x, pts4)

    return tuple(map(float, x))

# ------------------------- Single tetra & scene: ms solve -------------------------
def ms_on_tetra_for_scene(model_tmpl, first_vars, solver, tet_vertices, fverts_scene):
    """
    在一个四面体上，对单个场景：
      ms = min_{lambda>=0, 1^T lambda=1} [ obj_expr(K(lambda)) - sum_j lambda_j f(v_j) ]
    返回 (ms_value, lambda*, new_point)；若失败返回 (inf, None, None)
    """
    pairs = sorted(
        [(tuple(map(float, tet_vertices[j])), float(fverts_scene[j])) for j in range(4)],
        key=lambda kv: (kv[0][0], kv[0][1], kv[0][2])
    )
    tet_vertices = [kv[0] for kv in pairs]
    fverts_scene = [kv[1] for kv in pairs]

    m = model_tmpl.clone()
    Kp = m.find_component(first_vars[0].name)
    Ki = m.find_component(first_vars[1].name)
    Kd = m.find_component(first_vars[2].name)
    if any(v is None for v in (Kp, Ki, Kd)):
        raise RuntimeError("克隆模型中找不到 Kp/Ki/Kd")

    m.lam = pyo.Var(range(4), domain=pyo.NonNegativeReals)
    m.lam_sum = pyo.Constraint(expr=sum(m.lam[j] for j in range(4)) == 1.0)

    vx = [tet_vertices[j][0] for j in range(4)]
    vy = [tet_vertices[j][1] for j in range(4)]
    vz = [tet_vertices[j][2] for j in range(4)]
    m.link_kp = pyo.Constraint(expr=Kp == sum(m.lam[j]*vx[j] for j in range(4)))
    m.link_ki = pyo.Constraint(expr=Ki == sum(m.lam[j]*vy[j] for j in range(4)))
    m.link_kd = pyo.Constraint(expr=Kd == sum(m.lam[j]*vz[j] for j in range(4)))

    m.As = pyo.Var()
    m.As_def = pyo.Constraint(expr=m.As == sum(m.lam[j]*fverts_scene[j] for j in range(4)))

    if hasattr(m, 'obj'): 
        m.del_component('obj')
    m.obj = pyo.Objective(expr=m.obj_expr - m.As, sense=pyo.minimize)

    res = solver.solve(m, tee=False)
    ok = (res.solver.status == SolverStatus.ok) and \
         (res.solver.termination_condition in {
             TerminationCondition.optimal,
             TerminationCondition.locallyOptimal
         })
    if not ok:
        return float('inf'), None, None

    ms_val = float(pyo.value(m.obj))
    lam_star = np.array([pyo.value(m.lam[j]) for j in range(4)], dtype=float)

    eps = 1e-3
    lam_clip = np.maximum(lam_star, eps)
    lam_clip = lam_clip / lam_clip.sum()

    new_pt = np.dot(lam_clip, np.array(tet_vertices, dtype=float))
    return ms_val, lam_star, tuple(map(float, new_pt))

# ------------------------- Evaluate all tetrahedra -------------------------
def evaluate_all_tetra(nodes, scen_values, model_list, first_vars_list, solver):
    """
    nodes       : 当前节点列表（3D 点）
    scen_values : 形状 S×len(nodes)，每个场景在每个节点上的真实值
    返回 tri, per_tet； per_tet 每项含：
      'simplex_index','vert_idx','verts','fverts_sum','ms_per_scene','ms','LB','UB','x_ms_best_scene','best_scene','volume'
    """
    pts = np.asarray(nodes, dtype=float)
    if len(pts) < 4:
        return None, []
    tri = Delaunay(pts)  # simplices: (M,4)
    S = len(model_list)

    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    diam = float(np.linalg.norm(maxs - mins))

    vol_tol = 1e-12 * max(diam**3, 1.0)  # 体积阈值

    per_tet = []
    for k, simp in enumerate(tri.simplices):
        idxs = list(map(int, simp))
        verts = [tuple(pts[i]) for i in idxs]

        v0, v1, v2, v3 = np.array(verts)
        vol = abs(np.linalg.det(np.stack([v1 - v0, v2 - v0, v3 - v0], axis=1))) / 6.0
        if vol < vol_tol:
            continue

        fverts_per_scene = [[scen_values[ω][i] for i in idxs] for ω in range(S)]
        fverts_sum = [sum(fverts_per_scene[ω][j] for ω in range(S)) for j in range(4)]

        ms_scene = []
        xms_scene = []
        for ω in range(S):
            ms_val, lam_star, new_pt = ms_on_tetra_for_scene(
                model_list[ω], first_vars_list[ω], solver, verts, fverts_per_scene[ω]
            )
            ms_scene.append(ms_val)
            xms_scene.append(new_pt)

        if MS_AGG == "sum":
            ms_total = float(np.sum(ms_scene))
        elif MS_AGG == "mean":
            ms_total = float(np.mean(ms_scene))
        else:
            raise ValueError("MS_AGG must be 'sum' or 'mean'")

        LB = float(np.min(fverts_sum) + ms_total)
        UB = float(np.max(fverts_sum) + ms_total)

        best_scene = int(np.argmin(ms_scene))
        x_ms_best = xms_scene[best_scene]

        per_tet.append({
            "simplex_index": k,
            "vert_idx": idxs,
            "verts": verts,
            "fverts_sum": fverts_sum,
            "ms_per_scene": ms_scene,
            "ms": ms_total,
            "LB": LB,
            "UB": UB,
            "x_ms_best_scene": x_ms_best,
            "best_scene": best_scene,
            "volume": vol,
        })

    return tri, per_tet

# ------------------------- Pretty print -------------------------
def print_tetra_table(per_tet, active_mask, purple_set=None, prec=6):
    purple_set = set() if purple_set is None else set(purple_set)
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    tet_ids = [r["simplex_index"] for r in per_tet]
    active_set = {tid for tid in tet_ids if active_mask.get(tid, False)}

    def _mark(tid):
        s = f"T{tid}"
        flags = []
        if tid in active_set:  flags.append("*")
        if tid in purple_set:  flags.append("^")
        return s + ("".join(flags) if flags else "")

    header = ["row\\simp"] + [_mark(tid) for tid in tet_ids]
    rows = [
        ["UB"] + [f"{r['UB']:.{prec}f}" for r in per_tet],
        ["LB"] + [f"{r['LB']:.{prec}f}" for r in per_tet],
        ["ms"] + [f"{r['ms']:.3e}"       for r in per_tet],
    ]

    table = [header] + rows
    colw = [max(len(str(row[c])) for row in table) + 2 for c in range(len(header))]

    RED, PURPLE, RESET = "\033[31m", "\033[35m", "\033[0m"
    def colorize(col_idx, s):
        if col_idx == 0:
            return s
        tid = tet_ids[col_idx-1]
        if tid in purple_set:
            return f"{PURPLE}{s}{RESET}"
        elif tid in active_set:
            return f"{RED}{s}{RESET}"
        return s

    print("\n== Per-tetra summary ==")
    print("".join(colorize(c, str(header[c]).ljust(colw[c])) for c in range(len(header))))
    print("-"*sum(colw))
    for r in rows:
        line = []
        for c in range(len(header)):
            cell = str(r[c])
            pad  = cell.ljust(colw[c]) if c==0 else cell.rjust(colw[c])
            line.append(colorize(c, pad))
        print("".join(line))
    print("(红色列=active；紫色列=包含当前最小节点的单形；第1行=UB，第2行=LB，第3行=ms)\n")

def min_dist_to_nodes(pt, nodes):
    P = np.asarray(pt, float)
    X = np.asarray(nodes, float)
    return float(np.min(np.linalg.norm(X - P, axis=1)))

def print_per_scenario_ms(per_tet, max_scenarios_to_print=10, prec=3):
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    if not per_tet or "ms_per_scene" not in per_tet[0]:
        return
    S = len(per_tet[0]["ms_per_scene"])
    show = min(S, max_scenarios_to_print)
    head = "simp | " + " ".join([f"s{j}".rjust(10) for j in range(show)])
    print("== Per-tetra per-scenario ms (showing first", show, "of", S, "scenes) ==")
    print(head)
    print("-"*len(head))
    for r in per_tet:
        arr = r["ms_per_scene"][:show]
        sline = " ".join([f"{v:.{prec}e}".rjust(10) for v in arr])
        print(f"{r['simplex_index']:>4d} | {sline}")
    if show < S:
        print(f"... ({S-show} scenes omitted)")
    print()

# ------------------------- Plotly visualization -------------------------
def plot_iteration_plotly(iter_id, nodes, tri, active_mask, ub_node, next_node, per_tet):
    fig = go.Figure()
    nodes = np.asarray(nodes, float)

    fig.add_trace(go.Scatter3d(
        x=nodes[:, 0], y=nodes[:, 1], z=nodes[:, 2],
        mode='markers',
        marker=dict(size=4),
        name='nodes'
    ))

    if ub_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[ub_node[0]], y=[ub_node[1]], z=[ub_node[2]],
            mode='markers',
            marker=dict(size=6, symbol="circle", color="green"),
            name='current min node'
        ))

    if next_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[next_node[0]], y=[next_node[1]], z=[next_node[2]],
            mode='markers',
            marker=dict(size=6, symbol="circle", color="blue"),
            name='next node'
        ))

    if tri is not None:
        pts = tri.points
        legend_mesh_added = False
        legend_edge_added = False

        for r in per_tet:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False):
                continue

            idxs = r["vert_idx"]
            verts = np.array([pts[i] for i in idxs], dtype=float)

            I = [0, 0, 0, 1]
            J = [1, 1, 2, 2]
            K = [2, 3, 3, 3]

            fig.add_trace(go.Mesh3d(
                x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
                i=I, j=J, k=K,
                color="#ff7043",
                opacity=0.35,
                showscale=False,
                name="active simplex",
                showlegend=(not legend_mesh_added)
            ))
            legend_mesh_added = True

            edges = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
            for (a, b) in edges:
                pa, pb = verts[a], verts[b]
                fig.add_trace(go.Scatter3d(
                    x=[pa[0], pb[0]],
                    y=[pa[1], pb[1]],
                    z=[pa[2], pb[2]],
                    mode='lines',
                    line=dict(width=2, color="gray"),
                    name="active edge",
                    showlegend=(not legend_edge_added)
                ))
            legend_edge_added = True

        cent_x, cent_y, cent_z, texts = [], [], [], []
        for r in per_tet:
            v = np.mean(np.asarray(r["verts"]), axis=0)
            ms_strs = [f"{val:.2e}" for val in r["ms_per_scene"][:8]]
            more = "" if len(r["ms_per_scene"]) <= 8 else f" (+{len(r['ms_per_scene'])-8} more)"
            txt = (f"simp={r['simplex_index']}<br>"
                   f"LB={r['LB']:.6f}<br>UB={r['UB']:.6f}<br>"
                   f"ms={r['ms']:.3e}<br>"
                   f"ms_per_scene: [{', '.join(ms_strs)}]{more}")
            cent_x.append(v[0]); cent_y.append(v[1]); cent_z.append(v[2]); texts.append(txt)

        fig.add_trace(go.Scatter3d(
            x=cent_x, y=cent_y, z=cent_z,
            mode='markers',
            marker=dict(size=1, opacity=0.0),
            text=texts, hoverinfo="text",
            name="tetra info",
            showlegend=False
        ))

    fig.update_layout(
        title=f"Iteration {iter_id}",
        scene=dict(
            xaxis_title="Kp",
            yaxis_title="Ki",
            zaxis_title="Kd",
            aspectmode="cube"
        ),
        width=900,
        height=650,
        legend=dict(itemsizing="constant")
    )

    fig.show()

# ------------------------- Main loop -------------------------
def run_pid_simplex_3d(model_list, first_vars_list, solver, target_nodes=30,
                       min_dist=MIN_DIST, active_tol=ACTIVE_TOL, verbose=True):
    LB_hist, UB_hist, ms_hist, node_count = [], [], [], []
    UB_node_hist, add_node_hist = [], []

    ms_a_hist, ms_b_hist = [], []
    active_ratio_hist = []

    S = len(model_list)

    nodes = corners_from_var_bounds(first_vars_list[0])

    bounds_arr = np.array([[float(v.lb), float(v.ub)] for v in first_vars_list[0]], float)
    diam = float(np.linalg.norm(bounds_arr[:,1] - bounds_arr[:,0]))
    min_dist = float(min_dist)

    scen_values = [[None]*len(nodes) for _ in range(S)]
    for i, node in enumerate(nodes):
        for ω in range(S):
            scen_values[ω][i] = evaluate_Q_at(model_list[ω], first_vars_list[ω], node, solver)

    it = 0
    while len(nodes) < target_nodes:
        # 1) 全局 UB
        f_sum_per_node = [
            sum(scen_values[ω][i] for ω in range(S))
            for i in range(len(nodes))
        ]
        ub_idx = int(np.argmin(f_sum_per_node))
        UB_global = float(f_sum_per_node[ub_idx])
        UB_node = tuple(nodes[ub_idx])

        # 2) 评估所有四面体
        tri, per_tet = evaluate_all_tetra(
            nodes, scen_values, model_list, first_vars_list, solver
        )
        if tri is None or not per_tet:
            if verbose:
                print("Not enough nodes to make tetrahedra; stop.")
            break

        # 3) active mask
        active_mask = { r["simplex_index"]: (r["LB"] <= UB_global + active_tol) for r in per_tet }
        active = [r for r in per_tet if active_mask[r["simplex_index"]]]

        # 4) active ratio
        total_vol = sum(r["volume"] for r in per_tet)
        active_vol = sum(r["volume"] for r in per_tet if active_mask[r["simplex_index"]])
        active_ratio = active_vol / total_vol if total_vol > 0 else 0.0

        # 5) LB_global
        ms_adj = [r["ms"] for r in per_tet if ub_idx in r["vert_idx"]]
        ms_b = float(np.min(ms_adj)) if ms_adj else np.nan
        LB_global = UB_global + ms_b if ms_adj else float(np.min([r["LB"] for r in per_tet]))

        # 6) ms_a（active 内最小 ms）
        if any(active_mask.values()):
            ms_a = float(np.min([r["ms"] for r in per_tet if active_mask[r["simplex_index"]]]))
        else:
            ms_a = float(np.min([r["ms"] for r in per_tet]))
        ms_iter = ms_a

        # 7) 记录
        LB_hist.append(LB_global)
        UB_hist.append(UB_global)
        ms_hist.append(ms_iter)
        node_count.append(len(nodes))
        UB_node_hist.append(UB_node)

        ms_a_hist.append(ms_a)
        ms_b_hist.append(ms_b)
        active_ratio_hist.append(active_ratio)

        # 8) 打印
        simp_with_min = [r["simplex_index"] for r in per_tet if ub_idx in r["vert_idx"]]
        purple_set = set(simp_with_min)
        if verbose:
            print(f"[Iter {it}] Active simplex ratio = {active_ratio:.6f}")
            print(f"[Iter {it}] UB node {UB_node} is in simplices {sorted(simp_with_min)}")

        # 9) 候选池
        neighbor_active = [r for r in per_tet
                           if (r["simplex_index"] in purple_set) and active_mask.get(r["simplex_index"], False)]
        active_only     = [r for r in per_tet if active_mask.get(r["simplex_index"], False)]
        all_simplices   = list(per_tet)

        def score(cand):
            ms = cand["ms"]
            pt = cand.get("x_ms_best_scene", None)
            d  = (float('inf') if pt is None else min_dist_to_nodes(pt, nodes))
            return (ms, -d)

        pools = [
            ("neighbor-active", neighbor_active),
            ("active", active_only),
            ("all", all_simplices),
        ]

        if verbose:
            print(f"[Iter {it}] UB node {UB_node} is in simplices {sorted(simp_with_min)}")
            print(f"[Iter {it}] neighbor-active count = {len(neighbor_active)}, "
                  f"active count = {len(active_only)}, all = {len(all_simplices)}")

        # 10) 用 choose_point_in_simplex 在目标单形里落点（逐层回退）
        new_node = None
        chosen_ms = None
        chosen_from_pool = None
        chosen_simp = None

        points_np = np.asarray(nodes, float)
        scen_vals_np = np.asarray(scen_values, float).T  # (N, S)
        simplices_np = tri.simplices  # (M,4)

        for pool_name, pool in pools:
            if not pool:
                continue

            cand_sorted = sorted(pool, key=score)

            if verbose:
                topN = cand_sorted[:10]
                print(f"== ms candidates in [{pool_name}] (sorted by (ms, -dist)) ==")
                print(f"{'rank':>4} {'simp':>6} {'ms':>12}")
                print("-" * 60)
                for rnk, cand in enumerate(topN, start=1):
                    print(f"{rnk:>4} T{cand['simplex_index']:<4} {cand['ms']:>12.4e}")
                print()

            picked = False
            for rank, cand in enumerate(cand_sorted, start=1):
                sid = cand["simplex_index"]

                cand_pt = choose_point_in_simplex(
                    simplex_id=sid,
                    simplices=simplices_np,
                    points=points_np,
                    scen_values_per_point=scen_vals_np,  # (N,S)
                    mode="incenter",                      # 可切换 "farthest"/"face_centroid"/"barycenter"
                    min_dist=min_dist,
                    risk_mode="mean",
                    scenario_weights=None,
                    cvar_alpha=0.9,
                    rng=42,
                )

                if min_dist_to_nodes(cand_pt, nodes) >= min_dist:
                    new_node = cand_pt
                    chosen_ms = cand["ms"]
                    chosen_from_pool = pool_name
                    chosen_simp = sid
                    if verbose:
                        print(
                            f"Chosen node {tuple(map(float, cand_pt))} "
                            f"with ms={chosen_ms:.3e} "
                            f"(simp {sid}, rank #{rank}, pool={pool_name})"
                        )
                    picked = True
                    break
                else:
                    if verbose:
                        print(
                            f"Skip candidate {tuple(map(float, cand_pt))} "
                            f"(simp {sid}, rank #{rank}, pool={pool_name}) "
                            f"because too close to existing nodes (< {min_dist:g})."
                        )

            if picked:
                break

        if new_node is None:
            if verbose:
                print("New node too close for all candidates (or infeasible); stop.")
            break

        # 可视化
        plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, per_tet)

        # 加点并评估
        new_vals = []
        for ω in range(S):
            val = evaluate_Q_at(model_list[ω], first_vars_list[ω], new_node, solver)
            new_vals.append(val)

        nodes.append(tuple(map(float, new_node)))
        for ω in range(S):
            scen_values[ω].append(new_vals[ω])

        add_node_hist.append(new_node)
        it += 1

    return {
        "nodes": np.array(nodes, float),
        "LB_hist": LB_hist,
        "UB_hist": UB_hist,
        "ms_hist": ms_hist,
        "ms_a_hist": ms_a_hist,
        "ms_b_hist": ms_b_hist,
        "node_count": node_count,
        "UB_node_hist": UB_node_hist,
        "added_nodes": add_node_hist,
        "active_ratio_hist": active_ratio_hist,
    }

# ===================== MAIN =====================
RUN_QUICK_TEST = True  # True: 先用小规模验证

if RUN_QUICK_TEST:
    csv_path       = "data.csv"
    max_scenarios  = 1
    target_nodes   = 30
else:
    csv_path       = "data.csv"
    max_scenarios  = 99
    target_nodes   = 30

bounds = {
    "x":  (None, None),
    "u":  (None, None),
    "e":  (None, None),
    "I":  (-10, 10),
    "Kp": (0, 1),
    "Ki": (0, 1),
    "Kd": (0, 1),
}
weights = (1.0, 0.01)

# build models
model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)

# Gurobi solver
solver = pyo.SolverFactory('gurobi')
solver.options.update({
    'MIPGap': 1e-1,
    'NumericFocus': 1,
    'Presolve': 2,
    'NonConvex': 2,   # 必须
    'TimeLimit': 10,  # 可按需调整
})

# run
hist = run_pid_simplex_3d(
    model_list=model_list,
    first_vars_list=first_stg_vars_list,
    solver=solver,
    target_nodes=target_nodes,
    min_dist=MIN_DIST,
    active_tol=ACTIVE_TOL,
    verbose=True
)

print("\n==== Done ====")
print(f"Total nodes: {len(hist['nodes'])}")
print(f"Best UB: {min(hist['UB_hist']) if hist['UB_hist'] else None}")
print(f"Last LB: {hist['LB_hist'][-1] if hist['LB_hist'] else None}")


[Iter 0] Active simplex ratio = 1.000000
[Iter 0] UB node (1.0, 1.0, 1.0) is in simplices [2, 3]
[Iter 0] UB node (1.0, 1.0, 1.0) is in simplices [2, 3]
[Iter 0] neighbor-active count = 2, active count = 6, all = 6
== ms candidates in [neighbor-active] (sorted by (ms, -dist)) ==
rank   simp           ms
------------------------------------------------------------
   1 T2     -1.9957e-01
   2 T3     -1.9957e-01

Chosen node (0.7500000000000004, 0.4999999999999996, 0.7499999999999998) with ms=-1.996e-01 (simp 2, rank #1, pool=neighbor-active)


[Iter 1] Active simplex ratio = 1.000000
[Iter 1] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 8, 9, 10, 11]
[Iter 1] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 8, 9, 10, 11]
[Iter 1] neighbor-active count = 6, active count = 12, all = 12
== ms candidates in [neighbor-active] (sorted by (ms, -dist)) ==
rank   simp           ms
------------------------------------------------------------
   1 T9     -9.1991e-01
   2 T8     -9.1991e-01
   3 T4     -2.8997e-01
   4 T5     -2.8997e-01
   5 T10    -1.9957e-01
   6 T11    -1.9957e-01

Chosen node (0.6875, 0.3749999999999999, 0.9374999999999997) with ms=-9.199e-01 (simp 9, rank #1, pool=neighbor-active)


[Iter 2] Active simplex ratio = 0.989583
[Iter 2] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 10, 11, 12, 13, 16, 17]
[Iter 2] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 10, 11, 12, 13, 16, 17]
[Iter 2] neighbor-active count = 8, active count = 17, all = 18
== ms candidates in [neighbor-active] (sorted by (ms, -dist)) ==
rank   simp           ms
------------------------------------------------------------
   1 T17    -9.1991e-01
   2 T16    -9.1991e-01
   3 T10    -2.8997e-01
   4 T11    -2.8997e-01
   5 T12    -1.9957e-01
   6 T13    -1.9957e-01
   7 T5     -1.1924e-01
   8 T4     -4.9107e-02

Chosen node (0.671875, 0.34375, 0.9843749999999998) with ms=-9.199e-01 (simp 17, rank #1, pool=neighbor-active)


[Iter 3] Active simplex ratio = 0.989583
[Iter 3] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 8, 9, 14, 15, 16, 17, 20, 21]
[Iter 3] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 8, 9, 14, 15, 16, 17, 20, 21]
[Iter 3] neighbor-active count = 10, active count = 21, all = 22
== ms candidates in [neighbor-active] (sorted by (ms, -dist)) ==
rank   simp           ms
------------------------------------------------------------
   1 T20    -9.1991e-01
   2 T21    -9.1991e-01
   3 T14    -2.8997e-01
   4 T15    -2.8997e-01
   5 T16    -1.9957e-01
   6 T17    -1.9957e-01
   7 T9     -1.1924e-01
   8 T5     -1.1924e-01
   9 T8     -5.8090e-02
  10 T4     -4.9107e-02

Chosen node (0.66796875, 0.3359375, 0.9960937499999998) with ms=-9.199e-01 (simp 20, rank #1, pool=neighbor-active)


[Iter 4] Active simplex ratio = 0.989583
[Iter 4] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 6, 7, 8, 9, 18, 19, 20, 21, 24, 25]
[Iter 4] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 6, 7, 8, 9, 18, 19, 20, 21, 24, 25]
[Iter 4] neighbor-active count = 12, active count = 25, all = 26
== ms candidates in [neighbor-active] (sorted by (ms, -dist)) ==
rank   simp           ms
------------------------------------------------------------
   1 T25    -9.1991e-01
   2 T24    -9.1991e-01
   3 T18    -2.8997e-01
   4 T19    -2.8997e-01
   5 T20    -1.9957e-01
   6 T21    -1.9957e-01
   7 T7     -1.1924e-01
   8 T6     -1.1924e-01
   9 T5     -1.1924e-01
  10 T8     -6.0569e-02

Chosen node (0.6669921875, 0.333984375, 0.9990234374999998) with ms=-9.199e-01 (simp 25, rank #1, pool=neighbor-active)


...

[Iter 18] Active simplex ratio = 0.989583
[Iter 18] UB node (1.0, 1.0, 1.0) is in simplices [1, 5, 13, 14, 16, 17, 18, 19, 26, 27, 29, 35, 36, 37, 38, 40, 41, 44, 45, 48, 49, 50, 53, 54, 60, 61, 64, 66, 67, 68, 72, 73, 74, 80, 81, 84, 85]
[Iter 18] UB node (1.0, 1.0, 1.0) is in simplices [1, 5, 13, 14, 16, 17, 18, 19, 26, 27, 29, 35, 36, 37, 38, 40, 41, 44, 45, 48, 49, 50, 53, 54, 60, 61, 64, 66, 67, 68, 72, 73, 74, 80, 81, 84, 85]
[Iter 18] neighbor-active count = 37, active count = 85, all = 86
== ms candidates in [neighbor-active] (sorted by (ms, -dist)) ==
rank   simp           ms
------------------------------------------------------------
   1 T85    -2.8997e-01
   2 T84    -2.8997e-01
   3 T80    -1.9957e-01
   4 T81    -1.9957e-01
   5 T16    -1.1924e-01
   6 T36    -1.1924e-01
   7 T37    -1.1924e-01
   8 T26    -1.1924e-01
   9 T5     -1.1924e-01
  10 T1     -7.0283e-02

Chosen node (0.3336550407111645, 0.9163449630141258, 0.7499999962747099) with ms=-2.900e-01 (simp 85, rank

[Iter 19] Active simplex ratio = 0.989583
[Iter 19] UB node (1.0, 1.0, 1.0) is in simplices [1, 6, 9, 10, 11, 19, 20, 21, 22, 25, 27, 28, 30, 32, 38, 44, 45, 46, 48, 51, 52, 55, 56, 60, 61, 62, 65, 66, 68, 72, 73, 74, 78, 79, 80, 83, 84, 85, 86]
[Iter 19] UB node (1.0, 1.0, 1.0) is in simplices [1, 6, 9, 10, 11, 19, 20, 21, 22, 25, 27, 28, 30, 32, 38, 44, 45, 46, 48, 51, 52, 55, 56, 60, 61, 62, 65, 66, 68, 72, 73, 74, 78, 79, 80, 83, 84, 85, 86]
[Iter 19] neighbor-active count = 39, active count = 88, all = 89
== ms candidates in [neighbor-active] (sorted by (ms, -dist)) ==
rank   simp           ms
------------------------------------------------------------
   1 T85    -2.8997e-01
   2 T86    -2.8997e-01
   3 T83    -1.9957e-01
   4 T84    -1.9957e-01
   5 T48    -1.1924e-01
   6 T61    -1.1924e-01
   7 T60    -1.1924e-01
   8 T38    -1.1924e-01
   9 T6     -1.1924e-01
  10 T73    -6.1234e-02

Chosen node (0.583413760177791, 0.9790862407535316, 0.43749999906867765) with ms=-2.900e-01 

[Iter 20] Active simplex ratio = 0.989583
[Iter 20] UB node (1.0, 1.0, 1.0) is in simplices [0, 6, 10, 11, 13, 15, 17, 19, 20, 24, 27, 28, 33, 34, 38, 39, 41, 42, 43, 49, 50, 52, 53, 55, 59, 60, 61, 65, 66, 67, 72, 73, 76, 77, 83, 84, 85, 86, 87, 90, 91]
[Iter 20] UB node (1.0, 1.0, 1.0) is in simplices [0, 6, 10, 11, 13, 15, 17, 19, 20, 24, 27, 28, 33, 34, 38, 39, 41, 42, 43, 49, 50, 52, 53, 55, 59, 60, 61, 65, 66, 67, 72, 73, 76, 77, 83, 84, 85, 86, 87, 90, 91]
[Iter 20] neighbor-active count = 41, active count = 93, all = 94
== ms candidates in [neighbor-active] (sorted by (ms, -dist)) ==
rank   simp           ms
------------------------------------------------------------
   1 T91    -2.8997e-01
   2 T90    -2.8997e-01
   3 T86    -1.9957e-01
   4 T87    -1.9957e-01
   5 T33    -1.1924e-01
   6 T50    -1.1924e-01
   7 T49    -1.1924e-01
   8 T24    -1.1924e-01
   9 T6     -1.1924e-01
  10 T65    -6.1234e-02

Chosen node (0.39585344004444767, 0.9947715601883829, 0.6093749997671694) 

[Iter 21] Active simplex ratio = 0.989583
[Iter 21] UB node (1.0, 1.0, 1.0) is in simplices [0, 6, 8, 13, 14, 16, 18, 20, 22, 24, 25, 29, 32, 33, 38, 39, 43, 44, 46, 47, 48, 54, 55, 57, 58, 60, 64, 65, 66, 70, 71, 72, 77, 78, 81, 82, 88, 89, 90, 91, 92, 99, 100]
[Iter 21] UB node (1.0, 1.0, 1.0) is in simplices [0, 6, 8, 13, 14, 16, 18, 20, 22, 24, 25, 29, 32, 33, 38, 39, 43, 44, 46, 47, 48, 54, 55, 57, 58, 60, 64, 65, 66, 70, 71, 72, 77, 78, 81, 82, 88, 89, 90, 91, 92, 99, 100]
[Iter 21] neighbor-active count = 43, active count = 98, all = 99
== ms candidates in [neighbor-active] (sorted by (ms, -dist)) ==
rank   simp           ms
------------------------------------------------------------
   1 T99    -2.8997e-01
   2 T100   -2.8997e-01
   3 T91    -1.9957e-01
   4 T92    -1.9957e-01
   5 T38    -1.1924e-01
   6 T55    -1.1924e-01
   7 T54    -1.1924e-01
   8 T29    -1.1924e-01
   9 T6     -1.1924e-01
  10 T70    -6.1234e-02

Chosen node (0.598963360011112, 0.9986928900470957, 0.4023


==== Done ====
Total nodes: 30
Best UB: 0.3616264624871053
Last LB: 0.0716591189625223
